# 02 - Simulación con Euler

Integramos el sistema de EDOs con Euler explícito para la instancia de 4 ciudades y observamos el decrecimiento de la energía y la matriz de activaciones final.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.instances import FOUR_CITIES, distance_matrix_for
from src.model import HopfieldParameters, random_initial_potentials
from src.solvers import euler_integrate
from src.analysis import (analyze_convergence, extract_route,
                          route_as_cycle_string, route_length, validate)
from src.visualization import plot_energy, plot_activation_matrix, plot_route

In [ ]:
instance = FOUR_CITIES
d = distance_matrix_for(instance)
params = HopfieldParameters(dt=1e-5, steps=30_000)

u0 = random_initial_potentials(instance.n, params.u0, rng=np.random.default_rng(7))
result = euler_integrate(u0, d, params, record_every=100)

conv = analyze_convergence(result.energies)
route = extract_route(result.V_final)
print('Ruta:', route_as_cycle_string(route, instance),
      '| L =', round(route_length(route, d), 4))
print('Energía final:', round(result.final_energy, 4),
      '| monótona:', conv.is_monotonic)
print('Válida:', validate(result.V_final).is_permutation)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
plot_energy(result.times, result.energies, title='Energía E(t)', ax=axes[0])
plot_activation_matrix(result.V_final, labels=instance.labels,
                       title='Matriz V final', ax=axes[1])
plot_route(instance.coords, route, labels=instance.labels,
           title='Ruta del modelo', ax=axes[2])
plt.tight_layout()
plt.show()